# Sağlık Risk Skorlama - Örnek Notebook

Bu notebook, sağlık risk skorlama sisteminin kullanımını göstermektedir.

In [ ]:
import sys
sys.path.append('../src')

from risk_scoring import HealthRiskScoring
from drift_detection import ModelDriftDetector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Risk Skorlama Sistemini Başlatma

In [ ]:
# Risk skorlama sistemini oluştur
risk_system = HealthRiskScoring()

# Sentetik veri oluştur
df = risk_system.generate_sample_data(1000)
print(f"Veri boyutu: {df.shape}")
print(f"Yüksek risk oranı: %{df['high_risk'].mean()*100:.1f}")

# İlk 5 satırı göster
df.head()

## 2. Veri Keşfi

In [ ]:
# Veri dağılımlarını görselleştir
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

numeric_cols = ['age', 'bmi', 'blood_pressure_systolic', 'cholesterol', 'glucose', 'exercise_hours_week']

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{col} Dağılımı')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekans')

plt.tight_layout()
plt.show()

## 3. Model Eğitimi ve Değerlendirme

In [ ]:
# Veriyi hazırla
X_train, X_test, y_train, y_test = risk_system.prepare_data(df)

# Modelleri eğit
risk_system.train_models(X_train, y_train)

# Performansı değerlendir
results = risk_system.evaluate_models(X_test, y_test)

In [ ]:
# Özellik önemini görselleştir
risk_system.plot_feature_importance()

In [ ]:
# Confusion matrislerini göster
risk_system.plot_confusion_matrices(y_test, results)

## 4. Hasta Risk Tahmini Örnekleri

In [ ]:
# Yüksek riskli hasta profili
high_risk_patient = {
    'age': 70,
    'bmi': 32,
    'blood_pressure_systolic': 160,
    'blood_pressure_diastolic': 100,
    'cholesterol': 280,
    'glucose': 140,
    'smoking': 1,
    'exercise_hours_week': 0.5,
    'family_history': 1,
    'stress_level': 5
}

print("Yüksek Riskli Hasta:")
risk_pred = risk_system.predict_risk(high_risk_patient)
for key, value in risk_pred.items():
    print(f"{key}: {value:.3f}")

In [ ]:
# Düşük riskli hasta profili
low_risk_patient = {
    'age': 30,
    'bmi': 22,
    'blood_pressure_systolic': 110,
    'blood_pressure_diastolic': 70,
    'cholesterol': 180,
    'glucose': 85,
    'smoking': 0,
    'exercise_hours_week': 5,
    'family_history': 0,
    'stress_level': 2
}

print("Düşük Riskli Hasta:")
risk_pred = risk_system.predict_risk(low_risk_patient)
for key, value in risk_pred.items():
    print(f"{key}: {value:.3f}")

## 5. Model Drift Tespiti

In [ ]:
# Drift detector başlat
drift_detector = ModelDriftDetector()

# Referans veri ayarla
train_df = X_train.copy()
train_df['high_risk'] = y_train
drift_detector.set_reference_data(train_df)

# Drift içeren üretim verisi simüle et
production_data = drift_detector.generate_production_data(n_samples=200, drift_factor=0.3)

print("Üretim verisi oluşturuldu - Drift faktörü: 0.3")
production_data.head()

In [ ]:
# Drift testleri çalıştır
test_results = drift_detector.run_drift_tests(production_data)

## 6. Risk Skorlarının Dağılım Karşılaştırması

In [ ]:
# Test setinde risk skorları
X_test_scaled = risk_system.scaler.transform(X_test)
lr_scores = risk_system.lr_model.predict_proba(X_test_scaled)[:, 1]
rf_scores = risk_system.rf_model.predict_proba(X_test)[:, 1]

# Görselleştir
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression skorları
axes[0].hist(lr_scores[y_test == 0], bins=30, alpha=0.7, label='Düşük Risk', color='blue')
axes[0].hist(lr_scores[y_test == 1], bins=30, alpha=0.7, label='Yüksek Risk', color='red')
axes[0].set_title('Logistic Regression - Risk Skoru Dağılımı')
axes[0].set_xlabel('Risk Skoru')
axes[0].set_ylabel('Frekans')
axes[0].legend()

# Random Forest skorları
axes[1].hist(rf_scores[y_test == 0], bins=30, alpha=0.7, label='Düşük Risk', color='blue')
axes[1].hist(rf_scores[y_test == 1], bins=30, alpha=0.7, label='Yüksek Risk', color='red')
axes[1].set_title('Random Forest - Risk Skoru Dağılımı')
axes[1].set_xlabel('Risk Skoru')
axes[1].set_ylabel('Frekans')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Özet ve Sonuçlar

Bu notebook'ta şunları öğrendik:

1. **Risk Skorlama**: Logistic Regression ve Random Forest ile sağlık risk skorları hesapladık
2. **Model Karşılaştırması**: İki modelin performansını karşılaştırdık
3. **Özellik Önemi**: Hangi faktörlerin risk üzerinde en etkili olduğunu gördük
4. **Model Drift**: Evidently AI ile model performansındaki değişimleri tespit ettik
5. **Gerçek Zamanlı Tahmin**: Yeni hasta verileri için risk skorları hesapladık

### Öneriler:
- Modeli düzenli aralıklarla yeniden eğitin
- Drift tespiti için sürekli izleme sistemleri kurun
- Yüksek riskli hastaları öncelikli takibe alın
- Model tahminlerini klinisyen değerlendirmesi ile birleştirin